In [1]:
from sklearn.datasets import load_digits

In [2]:
dt = load_digits(n_class=3)

In [3]:
dt.keys()

dict_keys(['data', 'target', 'frame', 'feature_names', 'target_names', 'images', 'DESCR'])

In [6]:
dt.target

array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 0, 1, 1, 0, 0, 2, 2, 2, 0, 1, 2, 1,
       0, 2, 2, 0, 0, 1, 2, 1, 1, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 2, 0, 1,
       2, 0, 1, 2, 0, 0, 1, 1, 0, 0, 2, 2, 2, 0, 1, 2, 1, 0, 2, 2, 0, 0,
       1, 2, 1, 1, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0,
       0, 1, 1, 0, 0, 2, 2, 2, 0, 1, 2, 1, 0, 2, 2, 0, 0, 1, 2, 1, 1, 1,
       1, 0, 1, 2, 2, 2, 0, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 0, 1, 1, 0, 0,
       2, 2, 2, 0, 1, 2, 1, 0, 2, 2, 0, 0, 1, 2, 1, 1, 1, 1, 0, 1, 2, 2,
       2, 0, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 0, 1, 1, 0, 0, 2, 2, 2, 0, 1,
       2, 1, 0, 2, 2, 0, 0, 1, 2, 1, 1, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 2,
       0, 1, 2, 0, 1, 2, 0, 0, 1, 1, 0, 0, 2, 2, 2, 0, 1, 2, 1, 0, 2, 2,
       0, 0, 1, 2, 1, 1, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 2, 0, 1, 2, 0, 1,
       2, 0, 0, 1, 1, 0, 0, 2, 2, 2, 0, 1, 2, 1, 0, 2, 2, 0, 0, 1, 2, 1,
       1, 1, 1, 0, 1, 2, 2, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 0, 1, 1, 2,
       2, 0, 1, 2, 1, 0, 2, 2, 0, 0, 1, 2, 1, 1, 1,

In [8]:
dt.images[36]


array([[ 0.,  0.,  0.,  8., 15.,  8.,  0.,  0.],
       [ 0.,  0.,  3., 16., 12., 16.,  4.,  0.],
       [ 0.,  0.,  2., 10.,  1., 16.,  4.,  0.],
       [ 0.,  0.,  0.,  0.,  8., 14.,  0.,  0.],
       [ 0.,  0.,  0.,  9., 15.,  3.,  0.,  0.],
       [ 0.,  3., 16., 14.,  4.,  0.,  0.,  0.],
       [ 0.,  4., 15., 14.,  7.,  1.,  0.,  0.],
       [ 0.,  0.,  0.,  9., 12., 14.,  4.,  0.]])

In [9]:
X = dt.images
Y = dt.target

In [11]:
import torch
import torch.nn as nn
from torch.optim import Adam

In [12]:
X = torch.FloatTensor(X)
Y = torch.LongTensor(Y)

In [13]:
X.shape

torch.Size([537, 8, 8])

In [15]:
X = X.reshape(537,1,8,8)

In [16]:
class MyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
    self.maxm = nn.MaxPool2d(kernel_size=2, stride=2)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    self.relu = nn.ReLU()
    self.flatten = nn.Flatten()
    self.linear1 = nn.Linear(64*2*2, 128)
    self.linear2 = nn.Linear(128, 3)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.flatten(x)
    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)

    return x


In [17]:
model = MyNet()

In [18]:
loss = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr=0.001)

In [19]:
from torch.utils.data import TensorDataset, DataLoader

In [20]:
dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [21]:
model.train() # model.train() before training and model.eval() before inference
c=0
for epoch in range(100):
    c=c+1
    epoch_loss = 0.0
    for batch_X, batch_Y in dataloader:
        opt.zero_grad()
        yp = model(batch_X)
        ls = loss(yp, batch_Y)
        ls.backward()
        opt.step()
        epoch_loss += ls.item()

    if c%10==0:
      print(epoch_loss)

0.017489123507402837
0.005385359334468376
0.002340429536161537
0.0011972987576882588
0.0005706086039936054
0.00031646879335767153
0.00020700146023955313
0.00014919895147613715
0.00011157672292938514
8.213877902107924e-05


In [22]:
torch.save(model.state_dict(), 'model_weights.pth')